In [1]:
import sqlite3

def bootstrap_warehouse():
    conn = sqlite3.connect(":memory:")
    conn.execute("""
        CREATE TABLE departments (
            dept_id INTEGER PRIMARY KEY,
            dept_name TEXT NOT NULL
        )
    """)
    conn.execute("""
        CREATE TABLE financials (
            ticker TEXT PRIMARY KEY,
            company_name TEXT NOT NULL,
            dept_id INTEGER,
            sector TEXT,
            market_cap_usd REAL,
            revenue_usd REAL,
            operational_cost_usd REAL,
            quarter TEXT NOT NULL,
            approval_status TEXT DEFAULT 'approved',
            FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
        )
    """)
    depts = [(1, "Cloud"), (2, "Devices"), (3, "AI Research")]
    conn.executemany("INSERT INTO departments VALUES (?,?)", depts)

    rows = [
        ("MSFT",  "Microsoft Corp", 1, "Technology", 3_100_000_000_000,  62_000_000_000, 38_000_000_000, "Q3", "approved"),
        ("AAPL",  "Apple Inc",      2, "Technology", 3_400_000_000_000,  94_000_000_000, 55_000_000_000, "Q3", "approved"),
        ("GOOGL", "Alphabet Inc",   3, "Technology", 2_100_000_000_000,  84_000_000_000, 51_000_000_000, "Q3", "approved"),
        ("NVDA",  "NVIDIA Corp",    3, "Technology", 3_200_000_000_000,  30_000_000_000, 12_000_000_000, "Q3", "approved"),
        ("AMZN",  "Amazon.com Inc", 1, "Consumer",   1_900_000_000_000, 143_000_000_000,131_000_000_000, "Q3", "pending"),
    ]
    conn.executemany("INSERT INTO financials VALUES (?,?,?,?,?,?,?,?,?)", rows)
    conn.commit()
    return conn

conn = bootstrap_warehouse()

# This is the "Execute" step -- exact math, no guessing, no calculator needed.
cur = conn.cursor()
cur.execute("""
    SELECT SUM(revenue_usd - operational_cost_usd)
    FROM financials
    WHERE quarter = 'Q3' AND approval_status = 'approved'
""")
print(f"Net profit (approved Q3): ${cur.fetchone()[0]:,.0f}")


Net profit (approved Q3): $114,000,000,000


In [2]:
DDL = """
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL
);

CREATE TABLE financials (
    ticker TEXT PRIMARY KEY,
    company_name TEXT NOT NULL,
    dept_id INTEGER,
    sector TEXT,
    market_cap_usd REAL,
    revenue_usd REAL,
    operational_cost_usd REAL,
    quarter TEXT NOT NULL,
    approval_status TEXT DEFAULT 'approved',
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)  -- this line is everything
);
"""

FEW_SHOT_EXAMPLES = """
Query: "What was our total revenue in Q3?"
SQL: SELECT SUM(revenue_usd) FROM financials WHERE quarter = 'Q3' AND approval_status = 'approved';

Query: "What was our net profit in Q3?"
SQL: SELECT SUM(revenue_usd - operational_cost_usd) FROM financials WHERE quarter = 'Q3' AND approval_status = 'approved';

Query: "How many transactions are still pending approval?"
SQL: SELECT COUNT(*) FROM financials WHERE approval_status = 'pending';
"""

SYSTEM_PROMPT = f"""You are a deterministic SQL DBA. Your ONLY job is to translate the user's
question into a single, flawless, executable SQLite query.

=== DATABASE SCHEMA ===
{DDL}

=== BUSINESS LOGIC EXAMPLES (learn the corporate rules from these) ===
{FEW_SHOT_EXAMPLES}

=== RULES ===
1. Use ONLY the tables and columns explicitly defined above. Never assume a column exists.
2. If the question cannot be answered with this schema, output EXACTLY: ERROR: OUT_OF_SCOPE
3. Output ONLY the raw SQL query. No markdown code fences. No explanations. No greetings.
4. "Net profit" always means revenue_usd - operational_cost_usd, and always filter approval_status = 'approved'
   unless the user explicitly asks about pending or all transactions.

=== EXECUTABLE SQL: ==="""

print(SYSTEM_PROMPT)


You are a deterministic SQL DBA. Your ONLY job is to translate the user's
question into a single, flawless, executable SQLite query.

=== DATABASE SCHEMA ===

CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL
);

CREATE TABLE financials (
    ticker TEXT PRIMARY KEY,
    company_name TEXT NOT NULL,
    dept_id INTEGER,
    sector TEXT,
    market_cap_usd REAL,
    revenue_usd REAL,
    operational_cost_usd REAL,
    quarter TEXT NOT NULL,
    approval_status TEXT DEFAULT 'approved',
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)  -- this line is everything
);


=== BUSINESS LOGIC EXAMPLES (learn the corporate rules from these) ===

Query: "What was our total revenue in Q3?"
SQL: SELECT SUM(revenue_usd) FROM financials WHERE quarter = 'Q3' AND approval_status = 'approved';

Query: "What was our net profit in Q3?"
SQL: SELECT SUM(revenue_usd - operational_cost_usd) FROM financials WHERE quarter = 'Q3' AND approval_status = 'approved';

Que

In [3]:
import re

BLACKLIST = re.compile(r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|GRANT|REVOKE|EXEC|ATTACH|PRAGMA)\b", re.IGNORECASE)

def is_safe(sql: str):
    """Two checks: blacklist any write verb (word-boundary matched), whitelist SELECT/WITH starts."""
    stripped = sql.strip()
    if BLACKLIST.search(stripped):
        return False, "BLOCKED: contains a write/DDL keyword"
    if not stripped.upper().startswith(("SELECT", "WITH")):
        return False, "BLOCKED: does not start with SELECT or WITH"
    return True, "OK"

tests = [
    "SELECT * FROM financials WHERE ticker = 'MSFT'",
    "DELETE FROM financials WHERE dept_id = 1",
    "SELECT update_time FROM financials",          # column named update_time -- must NOT be blocked
    "  select revenue_usd from financials",         # lowercase, leading whitespace
    "WITH recent AS (SELECT * FROM financials) SELECT * FROM recent",
    "Robert'); DROP TABLE financials;--",           # Little Bobby Tables
]
for t in tests:
    ok, reason = is_safe(t)
    print(f"{'ALLOW' if ok else 'DENY '}  {reason:42s}  {t!r}")


ALLOW  OK                                          "SELECT * FROM financials WHERE ticker = 'MSFT'"
DENY   BLOCKED: contains a write/DDL keyword       'DELETE FROM financials WHERE dept_id = 1'
ALLOW  OK                                          'SELECT update_time FROM financials'
ALLOW  OK                                          '  select revenue_usd from financials'
ALLOW  OK                                          'WITH recent AS (SELECT * FROM financials) SELECT * FROM recent'
DENY   BLOCKED: contains a write/DDL keyword       "Robert'); DROP TABLE financials;--"


In [4]:
import sqlite3, os, tempfile

# Build a small file-backed DB (read-only mode requires an actual file, not :memory:)
db_path = os.path.join(tempfile.gettempdir(), "readonly_demo.db")
if os.path.exists(db_path):
    os.remove(db_path)

setup_conn = sqlite3.connect(db_path)
setup_conn.execute("CREATE TABLE financials (ticker TEXT, revenue_usd REAL)")
setup_conn.execute("INSERT INTO financials VALUES ('MSFT', 62000000000)")
setup_conn.commit()
setup_conn.close()

# The AI agent gets THIS connection -- opened with mode=ro. This is the real defense,
# not a mock: the OS/SQLite layer denies writes regardless of what SQL string arrives.
ai_agent_conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

cur = ai_agent_conn.cursor()
cur.execute("SELECT * FROM financials")
print("Read works fine:", cur.fetchall())

try:
    ai_agent_conn.execute("DELETE FROM financials")
    ai_agent_conn.commit()
    print("DELETE SUCCEEDED -- this would be very bad")
except sqlite3.OperationalError as e:
    print(f"DELETE blocked by the database engine itself: {e}")

os.remove(db_path)


Read works fine: [('MSFT', 62000000000.0)]
DELETE blocked by the database engine itself: attempt to write a readonly database


In [5]:
import os
from dotenv import  load_dotenv

load_dotenv()

True

In [6]:
import sqlite3
from langchain_openai import ChatOpenAI
from langchain_community.utilities import  SQLDatabase
from langchain_community.agent_toolkits import  create_sql_agent

db_path = "warehouse.db"

if os.path.exists(db_path):
    os.remove(db_path)

setup = sqlite3.connect(db_path)

/Users/abhishekbarve/learning/agentic-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/2r/045m6l_x5k35k1x7v4c2rgwh0000gn/T/ipykernel_75759/278515.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import  SQLDatabase


In [7]:
setup.execute(
    "CREATE TABLE departments (dept_id INTEGER PRIMARY KEY , dept_name TEXT NOT NULL)"
)

setup.execute(
    "CREATE TABLE financials ("
    "ticker TEXT PRIMARY KEY, company_name TEXT NOT NULL, dept_id INTEGER, sector TEXT, "
    "market_cap_usd REAL, revenue_usd REAL, operational_cost_usd REAL, quarter TEXT NOT NULL, "
    "approval_status TEXT DEFAULT \'approved\', "
    "FOREIGN KEY (dept_id) REFERENCES departments(dept_id))"
)


setup.executemany("INSERT INTO departments VALUES (?,?)", [(1, "Cloud"), (2, "Devices"), (3, "AI Research")])
setup.executemany(
    "INSERT INTO financials VALUES (?,?,?,?,?,?,?,?,?)",
    [
        ("MSFT", "Microsoft Corp", 1, "Technology", 3_100_000_000_000, 62_000_000_000, 38_000_000_000, "Q3", "approved"),
        ("AAPL", "Apple Inc", 2, "Technology", 3_400_000_000_000, 94_000_000_000, 55_000_000_000, "Q3", "approved"),
        ("GOOGL", "Alphabet Inc", 3, "Technology", 2_100_000_000_000, 84_000_000_000, 51_000_000_000, "Q3", "approved"),
        ("NVDA", "NVIDIA Corp", 3, "Technology", 3_200_000_000_000, 30_000_000_000, 12_000_000_000, "Q3", "approved"),
        ("AMZN", "Amazon.com Inc", 1, "Consumer", 1_900_000_000_000, 143_000_000_000, 131_000_000_000, "Q3", "pending"),
    ],
)

setup.commit()
setup.close()

In [9]:
db = SQLDatabase.from_uri(f"sqlite:///file:{db_path}?mode=ro&uri=true")

In [16]:
llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
)

In [17]:
agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="tool-calling",
    verbose=True,
    max_iterations=3
)

In [18]:
result = agent.invoke({
    "input": "Calculate net profit for Q3, only for approval transactions."
})

print(result)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


departments, financials
Invoking: `sql_db_schema` with `{'table_names': 'financials'}`



CREATE TABLE financials (
	ticker TEXT, 
	company_name TEXT NOT NULL, 
	dept_id INTEGER, 
	sector TEXT, 
	market_cap_usd REAL, 
	revenue_usd REAL, 
	operational_cost_usd REAL, 
	quarter TEXT NOT NULL, 
	approval_status TEXT DEFAULT 'approved', 
	PRIMARY KEY (ticker), 
	FOREIGN KEY(dept_id) REFERENCES departments (dept_id)
)

/*
3 rows from financials table:
ticker	company_name	dept_id	sector	market_cap_usd	revenue_usd	operational_cost_usd	quarter	approval_status
MSFT	Microsoft Corp	1	Technology	3100000000000.0	62000000000.0	38000000000.0	Q3	approved
AAPL	Apple Inc	2	Technology	3400000000000.0	94000000000.0	55000000000.0	Q3	approved
GOOGL	Alphabet Inc	3	Technology	2100000000000.0	84000000000.0	51000000000.0	Q3	approved
*/
Invoking: `sql_db_query_checker` with `{'query': "SELECT SUM(revenue_usd - operational_cos